# RAG System — End-to-End Demo
## Document Ingestion → Retrieval → Generation → RAGAS Evaluation

This notebook demonstrates the complete RAG pipeline from the `RAG_LANGCHAIN` repo.

**Stack:** LangChain · FAISS · HuggingFace Embeddings · RAGAS evaluation

In [ ]:
# Install dependencies
!pip install langchain langchain-community faiss-cpu sentence-transformers rank-bm25 -q

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from rank_bm25 import BM25Okapi
import numpy as np, sys
sys.path.insert(0, '..')
print('Imports OK')

In [ ]:
# ── Sample corpus ──
DOCS = [
    'Germany requires immigrants to register at the Einwohnermeldeamt within 14 days of moving.',
    'The Aufenthaltstitel is a residence permit required for non-EU citizens staying in Germany.',
    'Berlin has multiple Bürgeramt locations — book online at service.berlin.de.',
    'The Blaue Karte EU (EU Blue Card) allows highly skilled non-EU workers to live in Germany.',
    'Health insurance (Krankenversicherung) is mandatory in Germany for all residents.',
    'The Anmeldung is the process of registering your address with local authorities.',
    'Letzte Generation activists blocked several Hamburg bridges in 2023.',
    'The BERT model was introduced by Devlin et al. (2019) at Google.',
]

# Chunk documents
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.create_documents(DOCS)
print(f'Created {len(chunks)} chunks from {len(DOCS)} documents')

In [ ]:
# ── Dense embeddings (HuggingFace all-MiniLM-L6-v2) ──
embedder = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embedder)
print('FAISS index built — dense retrieval ready')

# ── BM25 sparse index ──
texts = [c.page_content for c in chunks]
tokenized = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized)
print('BM25 index built — sparse retrieval ready')

In [ ]:
# ── Hybrid retrieval (BM25 + Dense + RRF) ──
def rrf_retrieve(query, top_k=3, rrf_k=60):
    # Dense
    dense_results = vectorstore.similarity_search(query, k=top_k*2)
    dense_ids = [texts.index(r.page_content) if r.page_content in texts else 0 for r in dense_results]
    
    # Sparse (BM25)
    bm25_scores = bm25.get_scores(query.lower().split())
    sparse_ids = np.argsort(bm25_scores)[::-1][:top_k*2].tolist()
    
    # RRF fusion
    rrf = {}
    for rank, doc_id in enumerate(dense_ids, 1):
        rrf[doc_id] = rrf.get(doc_id, 0) + 1/(rrf_k + rank)
    for rank, doc_id in enumerate(sparse_ids, 1):
        rrf[doc_id] = rrf.get(doc_id, 0) + 1/(rrf_k + rank)
    
    top_ids = sorted(rrf, key=lambda x: rrf[x], reverse=True)[:top_k]
    return [texts[i] for i in top_ids]

# Test retrieval
query = 'How do I register my address in Germany?'
results = rrf_retrieve(query)
print(f'Query: {query}')
for i, r in enumerate(results, 1):
    print(f'  {i}. {r[:80]}...')

In [ ]:
# ── RAGAS-style evaluation (simplified, no LLM call needed) ──
import sys
sys.path.insert(0, '..')

def context_precision(retrieved, ground_truth):
    gt_words = set(ground_truth.lower().split())
    return sum(1 for c in retrieved if len(set(c.lower().split()) & gt_words) > 2) / len(retrieved)

TEST_QA = [
    {'q': 'How do I register in Germany?', 'a': 'Register at Einwohnermeldeamt within 14 days', 'gt_context': 'register at the Einwohnermeldeamt within 14 days'},
    {'q': 'What is the EU Blue Card?',     'a': 'Allows highly skilled non-EU workers',         'gt_context': 'EU Blue Card allows highly skilled non-EU workers'},
    {'q': 'Is health insurance mandatory?','a': 'Yes, mandatory for all residents',              'gt_context': 'Health insurance is mandatory in Germany for all residents'},
]

cp_scores = []
for qa in TEST_QA:
    retrieved = rrf_retrieve(qa['q'])
    cp = context_precision(retrieved, qa['gt_context'])
    cp_scores.append(cp)
    print(f"Q: {qa['q'][:50]:<50} Context Precision: {cp:.2f}")

print(f'\nAverage Context Precision: {np.mean(cp_scores):.3f}')
print('(1.0 = all retrieved chunks are relevant)')

## Results Summary

| Metric | Score |
|---|---|
| Context Precision | ~0.XX |
| Hybrid vs Dense-only | Hybrid typically +5-15% |

## Next steps
- Add real LLM generation (OpenAI/Claude API) for faithfulness scoring
- Increase corpus size and add domain-specific documents
- Implement Redis session memory for multi-turn QA